# Generate Calibration Store

This notebook generates model calibration metrics:
1. `overall_ae.json` - Overall A/E ratio
2. `yearly_ae.json` - A/E by observation year

**Output:** `knowledge_base/calibration/`

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import lightgbm as lgb

OUTPUT_DIR = '../../knowledge_base/calibration'
DATA_PATH = '../../data/ilec_cleaned.parquet'
MODEL_PATH = '../../models/lgbm_mortality_offset_poisson.txt'
YEAR_FACTORS_PATH = '../../models/year_factors_offset.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: ../../knowledge_base/calibration


## 1. Load Data and Model

In [ ]:
# Load data
df = pd.read_parquet(DATA_PATH)
print(f'Data shape: {df.shape}')

# Load model
model = lgb.Booster(model_file=MODEL_PATH)
print(f'Model loaded: {model.num_trees()} trees')

# Load year factors
year_factors_df = pd.read_csv(YEAR_FACTORS_PATH)
print(f'Year factors: {len(year_factors_df)} years')

Data shape: (45501036, 14)
Model loaded: 718 trees
Year factors: 8 years


In [ ]:
# Feature setup
NUMERICAL_FEATURES = ['Attained_Age', 'Issue_Age', 'Duration']
CATEGORICAL_FEATURES = [
    'Sex', 'Smoker_Status', 'Insurance_Plan', 'Face_Amount_Band',
    'Preferred_Class', 'SOA_Post_Lvl_Ind', 'SOA_Antp_Lvl_TP', 'SOA_Guar_Lvl_TP'
]
FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

# Prepare features
X = df[FEATURES].copy()
for col in CATEGORICAL_FEATURES:
    X[col] = X[col].astype('category')

print(f'Features: {len(FEATURES)}')

Features: 11


## 2. Calculate Predictions

In [ ]:
df['Predicted_Rate'] = model.predict(X)
df['Expected_Deaths'] = df['Predicted_Rate'] * df['Policies_Exposed']

print(f'Expected Deaths: {df["Expected_Deaths"].sum():,.0f}')
print(f'Actual Deaths: {df["Death_Count"].sum():,.0f}')

Expected Deaths: 4,557,142
Actual Deaths: 4,552,009


## 3. Overall A/E

In [ ]:
total_actual = df['Death_Count'].sum()
total_expected = df['Expected_Deaths'].sum()
overall_ae = total_actual / total_expected

if overall_ae > 1.05:
    interpretation = "Model underestimates mortality"
elif overall_ae < 0.95:
    interpretation = "Model overestimates mortality"
else:
    interpretation = "Model is well-calibrated"

print(f'Overall A/E: {overall_ae:.4f} - {interpretation}')

# Save
overall_ae_data = {
    "overall_ae": round(overall_ae, 4),
    "total_actual_deaths": int(total_actual),
    "total_expected_deaths": round(total_expected, 0),
    "interpretation": interpretation,
    "data_period": "2009-2019"
}

with open(f'{OUTPUT_DIR}/overall_ae.json', 'w') as f:
    json.dump(overall_ae_data, f, indent=2)
print('✓ overall_ae.json saved')

Overall A/E: 0.9989 - Model is well-calibrated
✓ overall_ae.json saved


## 4. Yearly A/E

In [ ]:
yearly = df.groupby('Observation_Year').agg({
    'Death_Count': 'sum',
    'Expected_Deaths': 'sum',
    'Policies_Exposed': 'sum'
})
yearly['ae_ratio'] = yearly['Death_Count'] / yearly['Expected_Deaths']

print('A/E by Year:')
display(yearly.round(4))

A/E by Year:


,Death_Count,Expected_Deaths,Policies_Exposed,ae_ratio
Observation_Year,,,,
2012,537286,542358.4244,5.103643e+07,0.9906
2013,554199,557018.7843,5.737303e+07,0.9949
2014,560393,562487.1167,5.755216e+07,0.9963
2015,565853,561111.3087,5.790785e+07,1.0085
2016,552127,558768.3243,5.792176e+07,0.9881
2017,558579,555209.2483,5.734615e+07,1.0061
2018,624284,613935.3701,6.303686e+07,1.0169
2019,599288,606253.8742,6.233901e+07,0.9885


In [ ]:
# Save
yearly_ae_data = {"by_year": {}, "year_factors": {}}

for year, row in yearly.iterrows():
    yearly_ae_data['by_year'][str(int(year))] = {
        "ae_ratio": round(row['ae_ratio'], 4),
        "actual_deaths": int(row['Death_Count']),
        "expected_deaths": round(row['Expected_Deaths'], 0)
    }

for _, row in year_factors_df.iterrows():
    yearly_ae_data['year_factors'][str(int(row['Year']))] = round(row['Year_Factor'], 4)

with open(f'{OUTPUT_DIR}/yearly_ae.json', 'w') as f:
    json.dump(yearly_ae_data, f, indent=2)
print('✓ yearly_ae.json saved')

✓ yearly_ae.json saved


## 5. Verify

In [ ]:
print('Generated files:')
for f in os.listdir(OUTPUT_DIR):
    if f.endswith('.json'):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
        print(f'  ✓ {f} ({size} bytes)')

Generated files:
  ✓ overall_ae.json (176 bytes)
  ✓ yearly_ae.json (1101 bytes)
